# NN from scratch
What if I try to make a nn from scratch? I followed no tutorial for this one :)

In [91]:
import numpy as np
import pandas as pd

training_data = pd.read_csv('data/train.csv').to_numpy()
test_data = pd.read_csv('data/test.csv').to_numpy()

In [128]:
X_train = training_data[:,1:]
Y_train = training_data[:,0].reshape(-1,1)
print(X_train.shape, Y_train.shape)

# normalise X
X_processed = X_train.T/255

# one hot encode Y
num_classes = 10
Y_processed = np.eye(num_classes)[Y_train.reshape(-1)].T
print(X_processed.shape, Y_processed.shape)

(42000, 784) (42000, 1)
(784, 42000) (10, 42000)


In [129]:
# initialise weights and biases
W1 = np.random.rand(10,784) - 0.5
W2 = np.random.rand(10,10) - 0.5
W3 = np.random.rand(10,10) - 0.5

b1 = np.random.rand(10,1) - 0.5
b2 = np.random.rand(10,1) - 0.5
b3 = np.random.rand(10,1) - 0.5

In [130]:
# Activaiton Functions
def ReLU(Z):
    return np.maximum(0, Z)

def dReLU(Z):
    return np.where(Z > 0, 1, 0)

def softmax(Z):
    exp_Z = np.exp(Z - np.max(Z))
    return exp_Z / np.sum(exp_Z, axis=0, keepdims=0)

In [147]:
# forward propagation
def forward_propagation(X, W1, b1, W2, b2, W3, b3):
    Z1 = W1 @ X + b1
    print("Z1:     ", Z1.shape)
    print("W1:     ", W1.shape)
    print("X:      ", X.shape)
    print("b1:     ", b1.shape)
    A1 = ReLU(Z1)
    print("A1:     ", A1.shape)
    print('\n')

    Z2 = W2 @ A1 + b2
    A2 = ReLU(Z2)

    Z3 = W3 @ A2 + b3
    A3 = softmax(Z3)

    return Z1, A1, Z2, A2, Z3, A3    

In [152]:
gradient_descent(X_processed, Y_processed, W1, b1, W2, b2, W3, b3, 0.01, 10000)

Z1:      (10, 10)
W1:      (10, 784)
X:       (784,)
b1:      (10, 1)
A1:      (10, 10)


delta3:  (10, 10)
A3:      (10, 10)
Y:       (10, 1)


delta2:  (10, 10)
W3.T:    (10, 10)
delta3:  (10, 10)


delta1:  (10, 10)
W2.T:    (10, 10)
delta2:  (10, 10)


ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 784 is different from 10)

In [149]:
# back propagation
def back_propagation(X, Y, Z1, A1, Z2, A2, Z3, A3):
    delta3 = A3 - Y[:,None] # loss function
    print("delta3: ", delta3.shape)
    print("A3:     ", A3.shape)
    print("Y:      ", Y[:,None].shape)
    dW3 = delta3 @ A2.T
    db3 = delta3
    print('\n')

    delta2 = (W3.T @ delta3) * dReLU(Z2)
    print("delta2: ", delta2.shape)
    print("W3.T:   ", W3.T.shape)
    print("delta3: ", delta3.shape)
    dW2 = delta2 @ A1.T
    db2 = delta2
    print('\n')

    delta1 = (W2.T @ delta2) * dReLU(Z1)
    print("delta1: ", delta1.shape)
    print("W2.T:   ", W2.T.shape)
    print("delta2: ", delta2.shape)
    dW1 = delta1 @ X.T.reshape(-1,1)
    print("X.T:    ", X.T.shape) 
    db1 = delta1

    return dW3, db3, dW2, db2, dW1, db1

In [150]:
def update_params(W3, b3, W2, b2, W1, b1, dW3, db3, dW2, db2, dW1, db1, alpha):
    W3 -= alpha * dW3
    b3 -= alpha * db3
    W2 -= alpha * dW2
    b2 -= alpha * db2
    W1 -= alpha * dW1
    b1 -= alpha * db1
    return W1, b1, W2, b2, W3, b3

In [151]:
# gradient descent
def gradient_descent(X, Y, W1, b1, W2, b2, W3, b3, alpha, samples):
    for sample in range(samples):
        Z1, A1, Z2, A2, Z3, A3 = forward_propagation(X[:,sample], W1, b1, W2, b2, W3, b3)
        dW3, db3, dW2, db2, dW1, db1 = back_propagation(X[:,sample], Y[:,sample], Z1, A1, Z2, A2, Z3, A3)
        W3, b3, W2, b2, W1, b1 = update_params(W3, b3, W2, b2, W1, b1, dW3, db3, dW2, db2, dW1, db1, alpha)
        print(-np.log(A3))